# Base model evaluation
In this notebook, the performance of selected base models will be evaluated on long texts.

**Selected models:**
- Encoder-only:
    - XLM-RoBERTa-large (FacebookAI/xlm-roberta-large) (0.6B parameters)
- Decoder-based:
    - Qwen3-Embedding-0.6B (Qwen/Qwen3-Embedding-0.6B) (0.6B parameters)

In [ ]:
!pip install mteb

ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
^C


In [1]:
import numpy as np

import torch
from datasets import load_dataset
import pandas as pd

import datasets

## Baseline models

In [2]:
from transformers import AutoTokenizer, AutoModel

In [3]:
def tokenize_chunking_strategy(tokenizer, inputs, chunk_size, overlap):
    real_chunks_size = chunk_size - tokenizer.num_special_tokens_to_add(pair=False)    # for each chunk special tokens will be appened after

    number_of_chunks = []    # number of chunks for each text in input
    outer_chunked_texts_batch = []    # long batch of all texts chunks

    for text in inputs:
        token_ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze()
        start = 0
        chunk_number = 0
        while start < len(token_ids):
            end = start + real_chunks_size
            chunk_tokens = token_ids[start:end]
            chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            outer_chunked_texts_batch.append(chunk_text)
            start += real_chunks_size - overlap
            chunk_number += 1
        number_of_chunks.append(chunk_number)

    tokenized_outer_batch = tokenizer(
        outer_chunked_texts_batch,
        add_special_tokens=True,
        max_length=chunk_size,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    return (tokenized_outer_batch, number_of_chunks)

def re_group_chunked_outputs(outputs_last_hidden_state, number_of_chunks):
    # outputs shape: [ num_chunks * num_texts, chunk_size, *]
    # converting to [num_texts, num_chunks, chunk_size, *]
    re_grouped = []
    text_starts_i = 0
    for n_chunks in number_of_chunks:
        text_ends_i = text_starts_i + n_chunks
        re_grouped.append(
            outputs_last_hidden_state[text_starts_i:text_ends_i, :, :]
        )
        text_starts_i = text_ends_i
    return re_grouped

def tokenize_first_startegy(tokenizer, inputs, max_length):
    tokenized = tokenizer(
        inputs,
        add_special_tokens=True,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return tokenized


def tokenize_max_tokens_strategy(tokenizer, inputs):
    tokenized = tokenizer(
        inputs,
        add_special_tokens=True,
        padding="longest",
        truncation=True,
        max_length=tokenizer.model_max_length,
        return_tensors="pt")
    return tokenized

## Models

In [ ]:
from mteb import EncoderProtocol
from mteb.similarity_functions import cos_sim

In [ ]:
from enum import Enum

class Strategy(Enum):
    chunking = "chunking"
    first = "first"
    max_tokens = "max_tokens"

In [ ]:
class XMLRoBERTa(EncoderProtocol):

    name = "xlm-roberta-large"
    similarity = staticmethod(cos_sim)

    def __init__(self, max_size, strategy: Strategy, overlap):
        self.model = AutoModel.from_pretrained(
            "FacebookAI/xlm-roberta-large",
            dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map="auto")

        self.tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-large")

        self.max_size = max_size
        self.strategy = strategy
        self.overlap = overlap

        self.model.eval()

    def __encode_batch(
            self,
            texts,
            **kwargs) -> np.ndarray:

        if self.strategy == Strategy.chunking:
            tokenized, numbers_of_chunks = tokenize_chunking_strategy(self.tokenizer, texts, self.max_size, self.overlap)
        if self.strategy == Strategy.first:
            tokenized = tokenize_first_startegy(self.tokenizer, texts, self.max_size)
        if self.strategy == Strategy.max_tokens:
            tokenized = tokenize_max_tokens_strategy(self.tokenizer, texts)

        with torch.inference_mode():
            tokenized = {k: v.to(self.model.device) for k, v in tokenized.items()}
            outputs = self.model(**tokenized)

            if self.strategy == Strategy.chunking:
                re_grouped = re_group_chunked_outputs(outputs.last_hidden_state, numbers_of_chunks)
                embeddings = torch.stack([
                    t.mean(dim=1).mean(dim=0).detach().cpu()
                    for t in re_grouped
                ])
            if (self.strategy == Strategy.first
                or self.strategy == Strategy.max_tokens):
                embeddings = outputs.last_hidden_state.mean(dim=1)


        embeddings = embeddings.detach().cpu().numpy()

        del outputs, tokenized
        torch.cuda.empty_cache()
        return embeddings

    def encode(
            self,
            inputs,
            *,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type,
            **kwargs):

        batch_size = kwargs["batch_size"]

        texts = [text for batch in inputs for text in batch["text"]]

        all_embeddings = []

        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            batch_embeddings = self.__encode_batch(batch)
            all_embeddings.append(batch_embeddings)

        return_embeddings = np.vstack(all_embeddings)
        print(return_embeddings.shape)
        return return_embeddings



class Qwen3_Embedding(EncoderProtocol):

    name = "Qwen3-Embedding-0.6B"
    similarity = staticmethod(cos_sim)

    def __init__(self, max_size, strategy, overlap, return_hidden_states=False):
        self.tokenizer = AutoTokenizer.from_pretrained(
            "Qwen/Qwen3-Embedding-0.6B",
            padding_side='left')

        self.model = AutoModel.from_pretrained(
            "Qwen/Qwen3-Embedding-0.6B",
            dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map="auto")

        self.max_size = max_size
        self.strategy = strategy
        self.overlap = overlap

        self.return_hidden_states = return_hidden_states

        self.model.eval()

    def __get_eos_token_embedding(self, last_hidden_states):
        return last_hidden_states[:, -1]

    def preprocess_query(self, texts):
        task = 'Given a search query, retrieve relevant passages that answer the query'
        return f'Instruct: {task}\nQuery:{texts}'

    def __encode_batch(self, texts, **kwargs) -> np.ndarray:
        if self.strategy == Strategy.chunking:
            tokenized, numbers_of_chunks = tokenize_chunking_strategy(
                self.tokenizer, texts, self.max_size, self.overlap
            )
        elif self.strategy == Strategy.first:
            tokenized = tokenize_first_startegy(self.tokenizer, texts, self.max_size)
        else:
            tokenized = tokenize_max_tokens_strategy(self.tokenizer, texts)

        with torch.inference_mode():
            tokenized = {k: v.to(self.model.device) for k, v in tokenized.items()}
            outputs = self.model(**tokenized)

            if self.strategy == Strategy.chunking:
                re_grouped = re_group_chunked_outputs(
                    outputs.last_hidden_state, numbers_of_chunks
                )
                embeddings = torch.stack([
                    self.__get_eos_token_embedding(t).mean(dim=0)
                    for t in re_grouped
                ])
            else:
                embeddings = self.__get_eos_token_embedding(
                    outputs.last_hidden_state
                )

        embeddings = embeddings.detach().cpu().numpy()

        del outputs, tokenized
        torch.cuda.empty_cache()
        return embeddings

    def encode(
            self,
            inputs,
            *,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type,
            **kwargs):

        batch_size = kwargs["batch_size"]

        texts = [text for batch in inputs for text in batch["text"]]
        if prompt_type.value == "query":
            texts = [self.preprocess_query(t) for t in texts]

        all_embeddings = []

        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            batch_embeddings = self.__encode_batch(batch)
            all_embeddings.append(batch_embeddings)

        return_embeddings = np.vstack(all_embeddings)
        print(return_embeddings.shape)
        return return_embeddings

## LongEmbed LEMBWikimQARetrieval

In [ ]:
import mteb

In [ ]:
long_embed_wiki_task = mteb.get_task("LEMBWikimQARetrieval")

In [ ]:
import json
import gc

strategies = [
    {
        "name": "Chunking",
        "strategy": Strategy.chunking,
        "max_size": 512,
        "overlap": 0
    },
    {
        "name": "Chunking 64 Overlap",
        "strategy": Strategy.chunking,
        "max_size": 512,
        "overlap": 64
    },
    {
        "name": "First",
        "strategy": Strategy.first,
        "max_size": 512,
        "overlap": None
    },
]

main_scores_roberta = []
roberta = None

for strategy in strategies:
    roberta = XMLRoBERTa(
        max_size=strategy["max_size"],
        strategy=strategy["strategy"],
        overlap=strategy["overlap"])

    eval = long_embed_wiki_task.evaluate(roberta, encode_kwargs={"batch_size": 4})
    main_scores_roberta.append(eval["default"]["main_score"])
    # save scores, eval is dictionary
    file_name = f"roberta_{strategy['name']}_scores.json"
    with open(file_name, "w") as f:
        json.dump(eval, f)

    # free memory just in case
    del roberta
    torch.cuda.empty_cache()
    gc.collect()

main_scores_qwen3 = []
qwen3 = None

for strategy in strategies:
    qwen3 = Qwen3_Embedding(
        max_size=strategy["max_size"],
        strategy=strategy["strategy"],
        overlap=strategy["overlap"])

    eval = long_embed_wiki_task.evaluate(qwen3, encode_kwargs={"batch_size": 4})
    main_scores_qwen3.append(eval["default"]["main_score"])
    # save scores, eval is dictionary
    file_name = f"qwen3_{strategy['name']}_scores.json"
    with open(file_name, "w") as f:
        json.dump(eval, f)

    # free memory just in case
    del qwen3
    torch.cuda.empty_cache()
    gc.collect()

In [ ]:
main_scores_roberta

In [ ]:
main_scores_qwen3

In [ ]:
pd.DataFrame(
    {
        "Strategy": [s["name"] for s in strategies],
        "Qwen3 score": main_scores_qwen3,
        "XML-RoBERTa score": main_scores_roberta
    }
).set_index("Strategy")

# Memory module

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [5]:
class Writer(nn.Module):

    def __init__(self, hidden_size):
        super().__init__()

        self.hidden_size = hidden_size

        # update attention Key and Query matrix
        self.W_k_update = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        self.W_q_update = nn.Linear(self.hidden_size, self.hidden_size, bias=False)

        # GRU matrices
        self.W_r = nn.Linear(self.hidden_size, self.hidden_size)
        self.U_r = nn.Linear(self.hidden_size, self.hidden_size)
        self.W_z = nn.Linear(self.hidden_size, self.hidden_size)
        self.U_z = nn.Linear(self.hidden_size, self.hidden_size)
        self.W_c = nn.Linear(self.hidden_size, self.hidden_size)
        self.U_c = nn.Linear(self.hidden_size, self.hidden_size)

    def get_update_attention_weights(self, h, memory):
        """
        h: hidden state (batch_size, hidden_state_dim)
        memory: memory (batch_size, memory_size, hidden_state_dim)
        """
        k = self.W_k_update(memory)    # (batch_size, memory_size, hidden_state_dim)
        q = self.W_q_update(h)    # (batch_size, hidden_state_dim)
        # casting to float32 just in case if the attention scores will be nan
        print(q.size())
        print(k.size())

        # expand to the same size as k
        expanded_q = q.unsqueeze(1).expand_as(k)
        print(expanded_q.size())

        scores = torch.bmm(k.float(), expanded_q.float()).squeeze(-1) / math.sqrt(self.hidden_size)  # (batch_size, memory_size)
        attention_weights = F.softmax(scores, dim=1)    # (batch_size, memory_size)
        return attention_weights.to(dtype=memory.dtype)

    def forward(self, input, memory):
        """
        input: input (batch_size, hidden_state_dim)
        memory: memory (batch_size, memory_size, hidden_state_dim)
        """
        # Attention
        attention_weights_upd = self.get_update_attention_weights(input, memory).unsqueeze(-1)

        # GRU
        input_expanded = input.unsqueeze(1).expand_as(memory)
        r = F.sigmoid(self.W_r(input_expanded) + self.U_r(memory))
        z = F.sigmoid(self.W_z(input_expanded) + self.U_z(memory))

        m_hat = F.tanh(self.W_c(input_expanded) + self.U_c(r * memory))
        updated_m = z * attention_weights_upd * m_hat + (1 - z * attention_weights_upd ) * memory
        return updated_m

class Memory(nn.Module):

    def __init__(self, hidden_size, memory_size, batch_size=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.memory_size = memory_size
        self.batch_size = batch_size
        init_freqs = 16
        self.init_freqs = init_freqs
        self.writer = Writer(hidden_size)

        self.init_mlp = nn.Sequential(
            nn.Linear(2 * init_freqs, hidden_size * 2),
            nn.GELU(),
            nn.Linear(hidden_size * 2, hidden_size),
        )

        self.register_buffer("memory_cells", torch.empty(0), persistent=False)
        self.reset(memory_size=memory_size, batch_size=batch_size)

        # reader will be only helpful for the token generation task.
        # for the embedding generation, only the writer will be used essentially.
        # self.reader = Reader(hidden_size)

    def _build_init_memory(self, memory_size, batch_size, device, dtype):
        pos = torch.linspace(-1.0, 1.0, steps=memory_size, device=device, dtype=dtype).unsqueeze(-1)
        freqs = torch.arange(1, self.init_freqs + 1, device=device, dtype=dtype).view(1, -1)  # (1, F)
        angles = math.pi * pos * freqs  # (M, F)
        feats = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)  # (M, 2F)
        base = self.init_mlp(feats)
        base = base / math.sqrt(self.hidden_size)
        mem = base.unsqueeze(0).expand(batch_size, -1, -1).contiguous()
        return mem

    def reset(self, memory_size, batch_size):
        self.memory_size = memory_size
        self.batch_size = batch_size

        device = self.writer.W_c.weight.device
        dtype = self.writer.W_c.weight.dtype

        self.memory_cells = self._build_init_memory(
            memory_size=self.memory_size,
            batch_size=self.batch_size,
            device=device,
            dtype=dtype,
        )

    def forward(self, input, token_mask, write_only=True):
        updated = self.writer(input, self.memory_cells)

        # apply update only where token is real
        token_mask = token_mask.to(dtype=torch.bool, device=updated.device)
        token_mask = token_mask.view(-1, 1, 1)

        self.memory_cells = torch.where(token_mask, updated, self.memory_cells)

        # if write_only:
        #     output = self.reader(input, self.memory_cells)
        #     return output
        return self.memory_cells


class MemoryPooling(nn.Module):

    def __init__(self, memory_size, hidden_size):
        """
        Pooling using the single head attention
        """
        super().__init__()
        self.memory_size = memory_size
        self.hidden_size = hidden_size
        self.W_k = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W_q = nn.Linear(hidden_size, hidden_size, bias=False)

        self.query = nn.Parameter(torch.randn(hidden_size))    # learnable query vector

        self.ff = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 4),
            nn.GELU(),
            nn.Linear(hidden_size * 4, hidden_size),
        )

    def forward(self, memory):
        batch_size = memory.shape[0]
        q = self.query.unsqueeze(0).expand(batch_size, -1)  # (B, D)
        K = self.W_k(memory)  # (B, N, D)
        q = self.W_q(q).unsqueeze(-1)  # (B, D, 1)

        scores = torch.bmm(K.float(), q.float()).squeeze(-1) / math.sqrt(self.hidden_size) # (B, N)
        attention_scores = F.softmax(scores, dim=1)  # (B, N)
        attention_scores = attention_scores.to(dtype=memory.dtype)

        pooled = torch.bmm(attention_scores.unsqueeze(1), memory).squeeze(1)  # (B, D)

        output = self.ff(pooled)
        return output


class TransformerWithMemory(nn.Module):

    def __init__(
            self,
            tokenizer,
            transformer,
            memory_size,
            chunk_size=512):
        super().__init__()
        self.tokenizer = tokenizer
        self.transformer = transformer
        self.chunk_size = chunk_size
        # transformr object can return a hidden states for all tokens
        self.memory_size = memory_size
        self.hidden_size = transformer.config.hidden_size

        self.memory_module = Memory(
            hidden_size = self.hidden_size,
            memory_size=memory_size,
            batch_size=1)

        self.memory_pooling = MemoryPooling(
            memory_size,
            self.hidden_size)

    def get_model_outputs(self, input):
        with torch.no_grad():
            outputs = self.transformer(**input)
            return outputs

    def get_hidden_states_per_batch(
            self,
            hidden_state,
            numbers_of_chunks,
            attention_mask):
        """
        hidden_state: (n_texts * n_chunks (varying), chunk_size, hidden_size)
        attention_mask: (n_texts * n_chunks (varying), chunk_size)

        numbers_of_chunks: list of numbers of chunks in each text

        output should be a :
            - list of (n_texts, chunk_size * max_n_chunks, hidden_size)
            - new attention mask
        """

        batch_size = len(numbers_of_chunks)
        print("len(numbers_of_chunks ) ", batch_size)
        chunk_size = self.chunk_size
        hidden_size = self.hidden_size

        device = hidden_state.device
        dtype = hidden_state.dtype

        max_n_chunks = max(numbers_of_chunks) if batch_size > 0 else 0
        max_len = chunk_size * max_n_chunks

        hidden_states_padded = torch.zeros((batch_size, max_len, hidden_size), device=device, dtype=dtype)
        attn_mask_padded = torch.zeros((batch_size, max_len), device=device, dtype=attention_mask.dtype)

        first = 0
        for b, n_chunks in enumerate(numbers_of_chunks):
            last = first + n_chunks

            # (n_chunks, chunk_size, hidden_size) -> (n_chunks*chunk_size, hidden_size)
            h_text = hidden_state[first:last, :, :].reshape(n_chunks * chunk_size, hidden_size)
            # (n_chunks, chunk_size) -> (n_chunks*chunk_size,)
            m_text = attention_mask[first:last, :].reshape(n_chunks * chunk_size)

            L = h_text.shape[0]
            hidden_states_padded[b, :L, :] = h_text
            attn_mask_padded[b, :L] = m_text
            first = last

        return hidden_states_padded, attn_mask_padded

    def write_memory(
            self,
            hidden_states,
            attention_mask):
        """
        hidden_states: (batch_size, seq_len)
        """
        batch_size, seq_len, _ = hidden_states.shape

        for i in range(seq_len):
            token_mask = attention_mask[:, i]
            memory_cells = self.memory_module(hidden_states[:, i, :], token_mask)

    def memory_pool(self):
        embed = self.memory_pooling(self.memory_module.memory_cells)
        return embed

    def encode(self, input, batch_size):

        tokenized, numbers_of_chunks = tokenize_chunking_strategy(
            self.tokenizer,
            input,
            self.chunk_size,
            0)

        tokenized = {k: v.to(self.transformer.device) for k, v in tokenized.items()}
        attention_mask = tokenized["attention_mask"]
        with torch.no_grad():
            outputs = self.get_model_outputs(tokenized)

        hidden_states = outputs.last_hidden_state.float()

        hidden_states_padded, attention_mask_padded = self.get_hidden_states_per_batch(
            hidden_states,
            numbers_of_chunks,
            attention_mask)

        self.write_memory(hidden_states_padded, attention_mask_padded)
        return self.memory_pool()

## Svaing and loading the memory parts


In [6]:
import torch
from pathlib import Path

def save_memory_parts(model, path: str):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "memory_module": model.memory_module.state_dict(),
            "memory_pooling": model.memory_pooling.state_dict(),
        },
        path
    )

def load_memory_parts(model, path: str, device="cpu", strict=True):
    ckpt = torch.load(path, map_location=device)

    missing_mm, unexpected_mm = model.memory_module.load_state_dict(
        ckpt["memory_module"], strict=strict
    )
    missing_mp, unexpected_mp = model.memory_pooling.load_state_dict(
        ckpt["memory_pooling"], strict=strict
    )

    return {
        "missing_memory_module": missing_mm,
        "unexpected_memory_module": unexpected_mm,
        "missing_memory_pooling": missing_mp,
        "unexpected_memory_pooling": unexpected_mp,
    }

## TransformerWithMemory from the hugging face transformers

In [7]:
from transformers import AutoTokenizer, AutoModel

def get_transformer_with_memory(
        model_name,
        chunk_size=512,
        memory_size=5):

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    taransformer = AutoModel.from_pretrained(
        model_name,
        dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map="auto")

    transformer_with_memory = TransformerWithMemory(
        tokenizer,
        taransformer,
        memory_size=memory_size)
    return transformer_with_memory

# Training

In [8]:
from datasets import load_dataset
from torch.utils.data import DataLoader

## MS MARCO dataset

In [9]:
class MSMARCOCollator:

    def __call__(self, batch):
        # batch_size is the number of rows in the batch
        # allign the batch as:
        # batch = {
        #      to_encode: (query1, doc_11, doc_12 ...
        #                  query2, doc_21, doc_22 ...
        #      sample_indexes: [],    # pairs of (start, end) of the sample
        #      pos_mask: []
        #      }

        batch_return = {
            "to_encode": [],
            "n_documents": [],
            "pos_mask": []
        }

        to_encode = []
        sample_indexes = []
        pos_mask = []

        query_index = 0
        for row in batch:
            to_encode.append(row["query"])
            to_encode.extend(row["passages"]["passage_text"])
            n_documents = len(row["passages"]["passage_text"])
            # pairs of (sample_start, sample)
            sample_indexes.append((query_index, n_documents + query_index + 1))
            query_index = n_documents + query_index + 1
            pos_mask.append(row["passages"]["is_selected"])

        return {
            "to_encode": to_encode,
            "sample_indexes": sample_indexes,
            "pos_mask": pos_mask
        }

ms_marco_collator = MSMARCOCollator()

ms_marco_dataset = load_dataset("microsoft/ms_marco", "v1.1", split="train")
# Filter out rows with zero positives (otherwise loss becomes -inf)
ms_marco_dataset = ms_marco_dataset.filter(lambda ex: 1 in [int(x) for x in ex["passages"]["is_selected"]])

ms_marco_dataloader = DataLoader(
    ms_marco_dataset,
    batch_size=4,
    collate_fn=ms_marco_collator
)

## Natural Questions dataset

In [10]:
class NaturalQuestionsCollator:

    def __call__(self, batch):

        to_encode = []
        sample_indexes = []
        query_index = 0
        for row in batch:
            to_encode.append(row["query"])
            to_encode.append(row["answer"])
            sample_indexes.append((query_index, query_index + 2))
            query_index += 2

        B = len(batch)
        pos_mask = torch.eye(B, dtype=torch.bool)

        return {
            "to_encode": to_encode,
            "sample_indexes": sample_indexes,
            "pos_mask": pos_mask
        }

natural_questions_collator = NaturalQuestionsCollator()

natural_questions_dataset = load_dataset(
    "sentence-transformers/natural-questions", split="train")

In [11]:
def get_query_doc_for_loss(batch, embeddings):
    sample_indexes = batch["sample_indexes"]
    device = embeddings.device

    pos_mask = batch["pos_mask"]
    doc_number_inconsistent = False
    pm_len = len(pos_mask[0])
    for i in range(len(pos_mask)):
        if len(pos_mask[i]) != pm_len:
            doc_number_inconsistent = True
            break
    if not doc_number_inconsistent:
        pos_mask = torch.as_tensor(batch["pos_mask"], device=device)
    else:
        for i in range(len(pos_mask)):
            pos_mask[i] = torch.as_tensor(batch["pos_mask"][i], device=device)
    q_embs = []
    p_embs = []
    pos_mask = []

    for i, (sample_start, sample_end) in enumerate(sample_indexes):
        q_embs.append(embeddings[sample_start])
        p_embs.append(embeddings[sample_start + 1:sample_end])
    return {
        "q_embs": q_embs,
        "p_embs": p_embs,
        "pos_mask": pos_mask
    }

# Training

In [1]:
from torch.amp import autocast, GradScaler

def multi_positive_infonce_batch_multiple_docs(q_embs, p_embs, pos_mask, tau=0.1):
    losses = []
    for i in range(len(q_embs)):
        q = q_embs[i]                  # (d,)
        p = p_embs[i]                  # (N, d)
        pos = torch.as_tensor(pos_mask[i], device=q.device).bool()

        p = p.to(q.device)

        # Guard: no candidates or no positives -> skip
        if p.numel() == 0 or pos.sum() == 0:
            continue

        # fp32 + normalize => logits in [-1, 1]/tau
        q = F.normalize(q.float(), dim=0)        # (d,)
        p = F.normalize(p.float(), dim=1)        # (N, d)

        logits = (p @ q) / tau                   # (N,) fp32

        # extra stability (optional but harmless)
        logits = logits - logits.max()

        log_probs = logits - torch.logsumexp(logits, dim=0)  # (N,) fp32
        losses.append(-log_probs[pos].mean())

    if len(losses) == 0:
        # Entire batch unusable
        print("Entire batch unusable")
        return torch.tensor(0.0, device=q_embs[0].device, requires_grad=True)

    return torch.stack(losses).mean()


def multi_positive_infonce_in_batch_negative(q_emb, p_emb, pos_mask, tau=0.05):
    q_emb = torch.stack(q_emb, dim=0)
    p_emb = torch.stack([e[0] for e in p_emb], dim=0).to(q_emb.device)

    # make everything fp32 for loss stability
    q = F.normalize(q_emb.float(), dim=1)
    p = F.normalize(p_emb.float(), dim=1)

    pos_mask = pos_mask.to(q.device).float()   # (B,B)
    denom = pos_mask.sum(dim=1)                # (B,)

    valid = denom > 0
    if not valid.any():
        # batch is unusable
        print("Batch is unusable - returning zeros")
        return torch.tensor(0.0, device=q.device, requires_grad=True)

    logits = (q @ p.T) / tau                   # (B,B) fp32
    log_probs = torch.log_softmax(logits, dim=1)

    loss_per_query = -(log_probs * pos_mask).sum(dim=1) / denom.clamp_min(1.0)
    return loss_per_query[valid].mean()


def train(
    stages,
    model,
    ckpt_saving_path,
    batch_size=8,
    steps_per_stage=1000,
):

    natural_questions_dataloader = DataLoader(
        natural_questions_dataset,
        batch_size=batch_size,
        collate_fn=natural_questions_collator
    )

    ms_marco_dataloader = DataLoader(
        ms_marco_dataset,
        batch_size=batch_size,
        collate_fn=ms_marco_collator
    )

    msmarco_iter = iter(ms_marco_dataloader)
    nq_iter = iter(natural_questions_dataloader)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    # freeze transformer
    for p in model.transformer.parameters():
        p.requires_grad = False

    model.transformer.eval()
    model.memory_module.train()
    model.memory_pooling.train()


    params = (
        list(model.memory_module.parameters())
        + list(model.memory_pooling.parameters())
    )

    optimizer = torch.optim.AdamW(params, lr=2e-5, weight_decay=0.01)
    optimizer.zero_grad(set_to_none=True)

    global_step = 1
    nq_loss_tracker = []
    ms_marco_loss_tracker = []

    for name, p in model.named_parameters():
        if p.requires_grad:
            if p.dtype == torch.float16:
                print("FP16 PARAM:", name)

    for stage in stages:
        print("Starting stage: ")
        print(stage)
        memory_size = stage["memory_size"]
        chunk_size = stage["chunk_size"]
        lr = stage["lr"]

        # update LR
        for g in optimizer.param_groups:
            g["lr"] = lr

        model.chunk_size = chunk_size

        accum_steps = 8

        msmarco_iter = iter(ms_marco_dataloader)
        nq_iter = iter(natural_questions_dataloader)

        for step_in_stage in range(steps_per_stage):

            running_loss = 0

            use_msmarco = (step_in_stage % 4 != 0)
            use_msmarco = True
            save_model = (global_step % 100 == 0)

            if save_model:
                save_memory_parts(model, ckpt_saving_path)
                print("Saved model")
                # print("     Latest loss: ")
                # print(f"     MS MARCO: {ms_marco_loss_tracker[-1]}")
                # print(f"     Natural Questions: {nq_loss_tracker[-1]}")

            if use_msmarco:
                try:
                    batch = next(msmarco_iter)
                except StopIteration:
                    msmarco_iter = iter(ms_marco_dataloader)
                    batch = next(msmarco_iter)
                dataset = "msmarco"
            else:
                try:
                    batch = next(nq_iter)
                except StopIteration:
                    nq_iter = iter(natural_questions_dataloader)
                    batch = next(nq_iter)
                dataset = "nq"


            n_documents = len(batch["to_encode"])
            model.memory_module.reset(memory_size, n_documents)
            embeddings = model.encode(batch["to_encode"], n_documents)
            outputs = get_query_doc_for_loss(batch, embeddings)


            if dataset == "msmarco":
                loss = multi_positive_infonce_batch_multiple_docs(
                    outputs["q_embs"],
                    outputs["p_embs"],
                    batch["pos_mask"],
                )
                ms_marco_loss_tracker.append(
                    {"step": global_step, "loss": loss.item()}
                )
            else:
                loss = multi_positive_infonce_in_batch_negative(
                    outputs["q_embs"],
                    outputs["p_embs"],
                    batch["pos_mask"],
                )
                nq_loss_tracker.append(
                    {"step": global_step, "loss": loss.item()}
                )

            loss = loss / accum_steps
            loss.backward()

            running_loss += loss.item()

            if (step_in_stage + 1) % accum_steps == 0:
                torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)


            global_step += 1

            if global_step % 4 == 0:
                print(f"Step: {global_step}", end=" ")
                print(f"loss: {running_loss / 4}")
                running_loss = 0


    return {
        "trackers": {
            "ms_marco_loss": ms_marco_loss_tracker,
            "nq_loss": nq_loss_tracker,
        },
        "ckpt_path": ckpt_saving_path
    }

KeyboardInterrupt: 

In [13]:
def first_nonfinite_param(named_params):
    for name, p in named_params:
        if p.requires_grad and p.data is not None and not torch.isfinite(p.data).all():
            return name
    return None

def first_nonfinite_grad(named_params):
    for name, p in named_params:
        if p.requires_grad and p.grad is not None and not torch.isfinite(p.grad).all():
            return name
    return None

# Quen3 + memory

In [14]:
path = "./quen3_memory"
transformer_name = "Qwen/Qwen3-Embedding-0.6B"

stages = [
    {"chunk_size": 512, "memory_size": 8, "lr": 1e-5},
    {"chunk_size": 256, "memory_size": 8, "lr": 1e-5},
    {"chunk_size": 64, "memory_size": 8, "lr": 1e-5},
    {"chunk_size": 32, "memory_size": 8, "lr": 1e-5},
    {"chunk_size": 64, "memory_size": 2, "lr": 1e-5},  # compression stress
]

model = get_transformer_with_memory(transformer_name)


In [15]:
strain_res = train(
    stages,
    model,
    path,
    batch_size=4,
    steps_per_stage=1000,
)

Starting stage: 
{'chunk_size': 512, 'memory_size': 8, 'lr': 1e-05}
len(numbers_of_chunks )  40
torch.Size([40, 1024])
torch.Size([4, 8, 1024])


RuntimeError: The expanded size of the tensor (4) must match the existing size (40) at non-singleton dimension 0.  Target sizes: [4, 8, 1024].  Tensor sizes: [40, 1, 1024]

In [24]:
model.memory_module.memory_cells.size()

torch.Size([4, 8, 1024])